# 量子神经网络(QNN)用于H2分子VMC计算 - 修复版

本notebook实现了一个量子神经网络(QNN)模型，用于H2分子的变分蒙特卡洛(VMC)计算。

主要特点：
- 量子电路支持批量输入
- 修复了PRNGKey序列化问题
- 修复了DynamicJaxprTracer类型错误
- 统一使用float32类型避免类型不匹配
- 经典-量子混合神经网络架构

In [ ]:
import jax
import jax.numpy as jnp
import pennylane as qml
from flax import nnx
from functools import partial
import numpy as np
import netket as nk
import netket.experimental as nkx
import matplotlib.pyplot as plt

# 禁用JAX的x64模式，使用float32
jax.config.update("jax_enable_x64", False)

In [ ]:
# 导入修复后的模型
from QNN_model_fixed import create_qnn_model

## H2分子系统设置

In [ ]:
from pyscf import gto, scf, fci

# 设置H2分子的几何构型
bond_length = 0.74  # H2平衡键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

# 创建分子对象，使用STO-3G基组
mol = gto.M(atom=geometry, basis='STO-3G')

# 进行Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"Hartree-Fock能量: {E_hf:.8f} Ha")

# 进行FCI计算作为参考
cisolver = fci.FCI(mf)
E_fci, fcivec = cisolver.kernel()
print(f"FCI能量: {E_fci:.8f} Ha")

# 使用NetKet创建哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

In [ ]:
# 定义希尔伯特空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,  # 总空间轨道数
    s=1/2,
    n_fermions_per_spin=(1, 1)  # 每种自旋的电子数
)

# 创建图结构
g = nk.graph.Graph(edges=[(0,1),(2,3)])

# 创建采样器
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)

## 创建QNN模型

In [ ]:
# 创建QNN模型
model_func, init_params = create_qnn_model(n_qubits=4, n_layers=2)

# 测试模型
test_input = jnp.array([[1,0,1,0]])
output = model_func(init_params, test_input)
print(f"模型测试输出: {output}")

## VMC计算设置

In [ ]:
# 使用JaxModel包装器
from netket.nn.models import JaxModel

# 创建NetKet兼容的模型
model = JaxModel(model_func, init_params)

# 创建变分状态
vstate = nk.vqs.MCState(sampler, model, n_samples=1008)
print(f"变分状态参数数量: {vstate.n_parameters}")

In [ ]:
# 设置优化器
opt = nk.optimizer.Sgd(learning_rate=0.05)
sr = nk.optimizer.SR(diag_shift=0.01)

# 创建VMC驱动器
gs = nk.driver.VMC(ha, opt, variational_state=vstate, preconditioner=sr)

# 添加日志记录器
exp_name = "h2_molecule_qnn_fixed"
gs.run(n_iter=300, out=exp_name)

## 结果分析

In [ ]:
# 读取结果
import json

# 加载能量数据
with open(f"{exp_name}.log", "r") as f:
    data = [json.loads(line) for line in f]

energies = [entry["Energy"]["mean"] for entry in data]
iterations = [entry["iteration"] for entry in data]

# 绘制能量收敛曲线
plt.figure(figsize=(10, 6))
plt.plot(iterations, energies, 'b-', label='QNN能量')
plt.axhline(y=E_hf, color='r', linestyle='--', label=f'Hartree-Fock: {E_hf:.6f} Ha')
plt.axhline(y=E_fci, color='g', linestyle='--', label=f'FCI: {E_fci:.6f} Ha')
plt.xlabel('迭代次数')
plt.ylabel('能量 (Ha)')
plt.title('H2分子能量收敛曲线')
plt.legend()
plt.grid(True)
plt.show()

print(f"最终能量: {energies[-1]:.8f} Ha")
print(f"与FCI能量差: {abs(energies[-1] - E_fci):.8f} Ha")